# EDA com Full Join das Abas

Este notebook carrega o Excel `BASE DE DADOS PEDE 2024 - DATATHON.xlsx`, faz um *full join* entre as abas `PEDE2022`, `PEDE2023`, `PEDE2024` pela chave `RA` e gera um EDA básico do resultado.

In [4]:
import pandas as pd

# ============================================================================
# CONFIGURAÇÕES
# ============================================================================

FILE_PATH = "/workspaces/Datathon-Machine-Learning-Engineering/data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

# ============================================================================
# 1. CARREGAR DADOS
# ============================================================================

print("📂 Carregando dados...")
df_2022 = pd.read_excel(FILE_PATH, sheet_name="PEDE2022")
df_2023 = pd.read_excel(FILE_PATH, sheet_name="PEDE2023")
df_2024 = pd.read_excel(FILE_PATH, sheet_name="PEDE2024")

print(f"  ✓ 2022: {df_2022.shape[0]:,} linhas x {df_2022.shape[1]} colunas")
print(f"  ✓ 2023: {df_2023.shape[0]:,} linhas x {df_2023.shape[1]} colunas")
print(f"  ✓ 2024: {df_2024.shape[0]:,} linhas x {df_2024.shape[1]} colunas")

📂 Carregando dados...
  ✓ 2022: 860 linhas x 42 colunas
  ✓ 2023: 1,014 linhas x 48 colunas
  ✓ 2024: 1,156 linhas x 50 colunas


In [5]:
COLUNAS_SEM_SUFIXO = ['RA']

# Função para adicionar sufixo nas colunas
def add_suffix_to_columns(df, suffix, exclude_cols):
    """
    Adiciona sufixo às colunas, exceto as que estão na lista de exclusão
    """
    new_columns = {}
    for col in df.columns:
        if col in exclude_cols:
            new_columns[col] = col  # Mantém o nome original
        else:
            new_columns[col] = f"{col}_{suffix}"  # Adiciona sufixo
    
    return df.rename(columns=new_columns)

# Renomeia cada base
df_2022_renamed = add_suffix_to_columns(df_2022, '2022', COLUNAS_SEM_SUFIXO)
df_2023_renamed = add_suffix_to_columns(df_2023, '2023', COLUNAS_SEM_SUFIXO)
df_2024_renamed = add_suffix_to_columns(df_2024, '2024', COLUNAS_SEM_SUFIXO)

print("  ✓ Colunas renomeadas")

  ✓ Colunas renomeadas


In [6]:

# Primeiro merge: 2022 + 2023
merged = df_2022_renamed.merge(
    df_2023_renamed,
    on='RA',
    how='outer'  # Full join - mantém todas as linhas
)

print(f"  ✓ Após merge 2022+2023: {merged.shape[0]:,} linhas x {merged.shape[1]} colunas")

# Segundo merge: (2022+2023) + 2024
merged = merged.merge(
    df_2024_renamed,
    on='RA',
    how='outer'  # Full join - mantém todas as linhas
)

print(f"  ✓ Após merge final: {merged.shape[0]:,} linhas x {merged.shape[1]} colunas")


  ✓ Após merge 2022+2023: 1,274 linhas x 89 colunas
  ✓ Após merge final: 1,661 linhas x 138 colunas


In [8]:
total_alunos = merged['RA'].nunique()
print(f"  • Total de alunos únicos (RA): {total_alunos:,}")

# Duplicatas (não deve ter)
duplicatas = merged['RA'].duplicated().sum()
print(f"  • Duplicatas de RA: {duplicatas}")

# Alunos por ano
alunos_2022 = df_2022_renamed['RA'].nunique()
alunos_2023 = df_2023_renamed['RA'].nunique()
alunos_2024 = df_2024_renamed['RA'].nunique()

print(f"\n  • Alunos em 2022: {alunos_2022:,}")
print(f"  • Alunos em 2023: {alunos_2023:,}")
print(f"  • Alunos em 2024: {alunos_2024:,}")

# Alunos que aparecem em todos os anos
alunos_todos_anos = set(df_2022['RA']) & set(df_2023['RA']) & set(df_2024['RA'])
print(f"  • Alunos presentes nos 3 anos: {len(alunos_todos_anos):,}")

# Alunos novos por ano
novos_2023 = set(df_2023['RA']) - set(df_2022['RA'])
novos_2024 = set(df_2024['RA']) - set(df_2023['RA'])
print(f"  • Alunos novos em 2023: {len(novos_2023):,}")
print(f"  • Alunos novos em 2024: {len(novos_2024):,}")

  • Total de alunos únicos (RA): 1,661
  • Duplicatas de RA: 0

  • Alunos em 2022: 860
  • Alunos em 2023: 1,014
  • Alunos em 2024: 1,156
  • Alunos presentes nos 3 anos: 468
  • Alunos novos em 2023: 414
  • Alunos novos em 2024: 391


In [9]:
print(merged.head())

print("\n📊 Info do dataset:")
print(f"  • Shape: {merged.shape}")
print(f"  • Colunas: {merged.shape[1]}")
print(f"  • Memória: {merged.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

        RA  Fase_2022 Turma_2022  Nome_2022  Ano nasc_2022  Idade 22_2022  \
0     RA-1        7.0          A    Aluno-1         2003.0           19.0   
1    RA-10        7.0          A   Aluno-10         2004.0           18.0   
2   RA-100        4.0          A  Aluno-100         2009.0           13.0   
3  RA-1000        NaN        NaN        NaN            NaN            NaN   
4  RA-1001        NaN        NaN        NaN            NaN            NaN   

  Gênero_2022  Ano ingresso_2022 Instituição de ensino_2022 Pedra 20_2022  \
0      Menina             2016.0             Escola Pública      Ametista   
1      Menina             2021.0             Escola Pública           NaN   
2      Menina             2019.0               Rede Decisão      Ametista   
3         NaN                NaN                        NaN           NaN   
4         NaN                NaN                        NaN           NaN   

   ... IPV_2024 IAN_2024          Fase Ideal_2024  Defasagem_2024  \
0  ..